In [2]:
import sys

sys.path.append("..")

import pandas as pd
import numpy as np

from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import ParameterGrid

from src.metrics import evaluate_predictions

In [3]:
X_train = pd.read_csv("../data/processed/X_train_normal.csv")
X_val = pd.read_csv("../data/processed/X_val.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_val = pd.read_csv("../data/processed/y_val.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [4]:
def best_threshold_by_f1(y_true, scores):

    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)

    best_index = np.argmax(f1_scores[:-1])

    return thresholds[best_index]

In [5]:
OCSVM_baseline = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01
)

OCSVM_baseline.fit(X_train)

val_scores = -OCSVM_baseline.score_samples(X_val)
test_scores = -OCSVM_baseline.score_samples(X_test)

val_metrics = evaluate_predictions(y_val, val_scores)
test_metrics = evaluate_predictions(y_test, test_scores)

print("Validation:")
print(val_metrics)

print("Test:")
print(test_metrics)

Validation:
{'precision': 0.1368421052631579, 'recall': 0.7878787878787878, 'f1': 0.23318385650224216, 'roc_auc': 0.9268138181491329, 'pr_auc': 0.2479053832573361}
Test:
{'precision': 0.14210526315789473, 'recall': 0.826530612244898, 'f1': 0.24251497005988024, 'roc_auc': 0.9573848954325681, 'pr_auc': 0.3445541474516531}


In [6]:
param_grid = {
    "kernel": ["rbf"],
    "nu": [0.001, 0.005, 0.01, 0.02],
    "gamma": ["scale", 0.001, 0.01]
}

results = []

best_f1 = 0
best_result = None

for params in ParameterGrid(param_grid):

    OCSVM_model = OneClassSVM(
        kernel=params["kernel"],
        nu=params["nu"],
        gamma=params["gamma"]
    )

    OCSVM_model.fit(X_train)

    val_scores = -OCSVM_model.score_samples(X_val)

    threshold = best_threshold_by_f1(y_val, val_scores)

    metrics = evaluate_predictions(
        y_true=y_val,
        scores=val_scores,
        threshold=threshold
    )

    result = {
        "kernel": params["kernel"],
        "nu": params["nu"],
        "gamma": params["gamma"],
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"]
    }

    results.append(result)

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_result = result

print("BEST RESULT:")
print(best_result)

BEST RESULT:
{'kernel': 'rbf', 'nu': 0.001, 'gamma': 0.01, 'threshold': np.float64(-0.7536032653591321), 'precision': 0.46938775510204084, 'recall': 0.46464646464646464, 'f1': 0.467005076142132, 'roc_auc': 0.9140892774890279, 'pr_auc': 0.2900425166015173}


In [7]:
results_df = pd.DataFrame(results)

top10 = results_df.sort_values(by="f1", ascending=False).head(10)

top10

,kernel,nu,gamma,threshold,precision,recall,f1,roc_auc,pr_auc
8,rbf,0.001,0.01,-0.753603,0.469388,0.464646,0.467005,0.914089,0.290043
0,rbf,0.001,scale,-0.062248,0.416667,0.505051,0.456621,0.926409,0.273173
1,rbf,0.005,scale,-0.316710,0.416667,0.505051,0.456621,0.926409,0.272220
2,rbf,0.010,scale,-0.688936,0.367647,0.505051,0.425532,0.926814,0.247905
3,rbf,0.020,scale,-1.127337,0.341772,0.545455,0.420233,0.931859,0.253285
9,rbf,0.005,0.01,-7.582591,0.312500,0.505051,0.386100,0.918216,0.224605
10,rbf,0.010,0.01,-12.538237,0.326241,0.464646,0.383333,0.924182,0.229006
11,rbf,0.020,0.01,-13.694840,0.311475,0.383838,0.343891,0.934268,0.199909
5,rbf,0.005,0.001,-301.533699,0.214286,0.393939,0.277580,0.929114,0.168785
6,rbf,0.010,0.001,-740.703261,0.201031,0.393939,0.266212,0.935418,0.160998


In [8]:
OCSVM_best = OneClassSVM(
    kernel="rbf",
    nu=0.001,
    gamma=0.01
)

OCSVM_best.fit(X_train)

test_scores = -OCSVM_best.score_samples(X_test)

test_metrics = evaluate_predictions(
    y_true=y_test,
    scores=test_scores,
    threshold=-0.7536032653591321
)

print(test_metrics)

{'precision': 0.49523809523809526, 'recall': 0.5306122448979592, 'f1': 0.5123152709359606, 'roc_auc': 0.9437706005305893, 'pr_auc': 0.3980266526346535}
